In [1]:
import pandas as pd
import numpy as np
import re
import spacy



In [2]:
#reading the dataset
data=pd.read_csv('amazon.csv')

In [3]:
#exploring the dataset
print(data.shape)

data.info()
data.describe()



(1465, 16)
<class 'pandas.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   product_id           1465 non-null   str  
 1   product_name         1465 non-null   str  
 2   category             1465 non-null   str  
 3   discounted_price     1465 non-null   str  
 4   actual_price         1465 non-null   str  
 5   discount_percentage  1465 non-null   str  
 6   rating               1465 non-null   str  
 7   rating_count         1463 non-null   str  
 8   about_product        1465 non-null   str  
 9   user_id              1465 non-null   str  
 10  user_name            1465 non-null   str  
 11  review_id            1465 non-null   str  
 12  review_title         1465 non-null   str  
 13  review_content       1465 non-null   str  
 14  img_link             1465 non-null   str  
 15  product_link         1465 non-null   str  
dtypes: str(16)
memory usage:

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
count,1465,1465,1465,1465,1465,1465,1465,1463,1465,1465,1465,1465,1465,1465,1465,1465
unique,1351,1337,211,550,449,92,28,1143,1293,1194,1194,1194,1194,1212,1412,1465
top,B07JW9H4J1,"Fire-Boltt Ninja Call Pro Plus 1.83"" Smart Wat...",Computers&Accessories|Accessories&Peripherals|...,₹199,₹999,50%,4.1,"9,378",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AHIKJUDTVJ4T6DV6IUGFYZ5LXMPA,AE55KTFVNXYFD5FPY...","$@|\|TO$|-|,Sethu madhav,Akash Thakur,Burger P...","R3F4T5TRYPTMIG,R3DQIEC603E7AY,R1O4Z15FD40PV5,R...","Worked on iPhone 7 and didn’t work on XR,Good ...","I am not big on camera usage, personally. I wa...",https://m.media-amazon.com/images/I/413sCRKobN...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
freq,3,5,233,53,120,56,244,9,6,10,10,10,10,8,3,1


In [4]:
data.isnull().sum()

product_id             0
product_name           0
category               0
discounted_price       0
actual_price           0
discount_percentage    0
rating                 0
rating_count           2
about_product          0
user_id                0
user_name              0
review_id              0
review_title           0
review_content         0
img_link               0
product_link           0
dtype: int64

In [5]:
#Dropping null values
data = data.dropna(subset=["rating_count"])
data.shape

(1463, 16)

In [6]:
#Removing the currency symbol and converting column to float
cols = ["discounted_price", "actual_price", "rating_count"]
for c in cols:
    data[c] = (
        data[c].str.replace("₹", "", regex=False)
        .str.replace(",", "", regex=False)
        .astype(float)
    )

In [7]:
data.describe()

,discounted_price,actual_price,rating_count
count,1463.000000,1463.000000,1463.000000
mean,3129.277122,5451.068544,18295.541353
std,6948.222850,10881.018448,42753.864952
min,39.000000,39.000000,2.000000
25%,325.000000,800.000000,1186.000000
50%,799.000000,1690.000000,5179.000000
75%,1999.000000,4312.500000,17336.500000
max,77990.000000,139900.000000,426973.000000


In [8]:
#Removing the percentage symbol from discount percentage
data["discount_percentage"] = data["discount_percentage"].str.replace("%", "", regex=False)

In [9]:
#Converting numerical object type to integer and float
data["discount_percentage"] = data["discount_percentage"].astype(int)

In [10]:
data["rating"] = (
    data["rating"]
    .str.replace(r"[^\d.]", "", regex=True)   # keep only digits + dot
    .replace("", np.nan)                      # convert empty strings to NaN
    .astype(float)
)

# Drop rows where rating is missing
data = data.dropna(subset=["rating"])

In [11]:
#cleaning the category column by removing the extra information in brackets
data['main_category'] = data['category'].str.split('|').str[0]

In [12]:
category_table = (
    data['main_category']
      .value_counts()
      .rename_axis('category')
      .reset_index(name='frequency')
)

category_table['percent (%)'] = (
    category_table['frequency'] / category_table['frequency'].sum() * 100
).round(2)

category_table

,category,frequency,percent (%)
0,Electronics,526,35.98
1,Computers&Accessories,451,30.85
2,Home&Kitchen,447,30.57
3,OfficeProducts,31,2.12
4,MusicalInstruments,2,0.14
5,HomeImprovement,2,0.14
6,Toys&Games,1,0.07
7,Car&Motorbike,1,0.07
8,Health&PersonalCare,1,0.07


In [13]:
# Select only the columns needed for the prototype
selected_columns = [
    "product_id",
    "product_name",
    "category",
    "main_category",
    "discounted_price",
    "actual_price",
    "discount_percentage",
    "rating",
    "rating_count",
    "review_title",
    "review_content"
]

df = data[selected_columns].copy()

df.head()

,product_id,product_name,category,main_category,discounted_price,actual_price,discount_percentage,rating,rating_count,review_title,review_content
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,Computers&Accessories,399.0,1099.0,64,4.2,24269.0,"Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,Computers&Accessories,199.0,349.0,43,4.0,43994.0,"A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,Computers&Accessories,199.0,1899.0,90,3.9,7928.0,"Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a..."
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,Computers&Accessories,329.0,699.0,53,4.2,94363.0,"Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou..."
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,Computers&Accessories,154.0,399.0,61,4.2,16905.0,"As good as original,Decent,Good one for second...","Bought this instead of original apple, does th..."


In [14]:
# Create additional pricing features
df["discount_amount"] = df["actual_price"] - df["discounted_price"]

df["discount_group"] = pd.cut(
    df["discount_percentage"],
    bins=[0, 15, 30, 50, 100],
    labels=["0-15%", "15-30%", "30-50%", "50%+"],
    include_lowest=True
)

df[["actual_price", "discounted_price", "discount_percentage", "discount_amount", "discount_group"]].head()

,actual_price,discounted_price,discount_percentage,discount_amount,discount_group
0,1099.0,399.0,64,700.0,50%+
1,349.0,199.0,43,150.0,30-50%
2,1899.0,199.0,90,1700.0,50%+
3,699.0,329.0,53,370.0,50%+
4,399.0,154.0,61,245.0,50%+


In [15]:
def clean_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)        # remove URLs
    text = re.sub(r"<.*?>", "", text)                 # remove HTML tags
    text = re.sub(r"[^a-zA-Z0-9\s.,!?']", " ", text)  # remove unusual characters
    text = re.sub(r"\s+", " ", text).strip()          # remove extra spaces
    
    return text

df["clean_review"] = df["review_content"].apply(clean_text)

df[["review_content", "clean_review"]].head()

,review_content,clean_review
0,Looks durable Charging is fine tooNo complains...,looks durable charging is fine toono complains...
1,I ordered this cable to connect my phone to An...,i ordered this cable to connect my phone to an...
2,"Not quite durable and sturdy,https://m.media-a...","not quite durable and sturdy, good, nice produ..."
3,"Good product,long wire,Charges good,Nice,I bou...","good product,long wire,charges good,nice,i bou..."
4,"Bought this instead of original apple, does th...","bought this instead of original apple, does th..."


In [16]:
def split_into_sentences(text):
    if pd.isna(text) or text == "":
        return []
    
    sentences = re.split(r'(?<=[.!?])\s+', text)
    sentences = [s.strip() for s in sentences if len(s.strip()) > 0]
    
    return sentences

df["review_sentences"] = df["clean_review"].apply(split_into_sentences)

df[["clean_review", "review_sentences"]].head()

,clean_review,review_sentences
0,looks durable charging is fine toono complains...,[looks durable charging is fine toono complain...
1,i ordered this cable to connect my phone to an...,[i ordered this cable to connect my phone to a...
2,"not quite durable and sturdy, good, nice produ...","[not quite durable and sturdy, good, nice prod..."
3,"good product,long wire,charges good,nice,i bou...","[good product,long wire,charges good,nice,i bo..."
4,"bought this instead of original apple, does th...","[bought this instead of original apple, does t..."


In [17]:
aspect_keywords = {
    "battery": ["battery", "charge", "charging", "power", "backup"],
    "quality": ["quality", "build", "material", "cheap", "premium"],
    "performance": ["performance", "speed", "fast", "slow", "lag", "smooth"],
    "durability": ["durable", "durability", "broke", "broken", "stopped working", "lasted"],
    "delivery": ["delivery", "shipping", "arrived", "package", "packaging"],
    "value": ["price", "worth", "value", "money", "cost", "expensive", "cheap"],
    "sound": ["sound", "bass", "volume", "audio", "noise"]
}

In [18]:
def detect_aspects(sentence, aspect_keywords):
    detected = []
    
    for aspect, keywords in aspect_keywords.items():
        for keyword in keywords:
            if keyword in sentence:
                detected.append(aspect)
                break
    
    return detected

In [19]:
absa_rows = []

for idx, row in df.iterrows():
    for sentence in row["review_sentences"]:
        detected_aspects = detect_aspects(sentence, aspect_keywords)
        
        for aspect in detected_aspects:
            absa_rows.append({
                "product_id": row["product_id"],
                "product_name": row["product_name"],
                "main_category": row["main_category"],
                "discount_percentage": row["discount_percentage"],
                "discount_group": row["discount_group"],
                "rating": row["rating"],
                "sentence": sentence,
                "aspect": aspect
            })

absa_df = pd.DataFrame(absa_rows)

absa_df.head()

,product_id,product_name,main_category,discount_percentage,discount_group,rating,sentence,aspect
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,looks durable charging is fine toono complains...,battery
1,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,looks durable charging is fine toono complains...,quality
2,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,looks durable charging is fine toono complains...,performance
3,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,looks durable charging is fine toono complains...,durability
4,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,the charging speed is slower than the original...,battery


In [20]:
absa_df["aspect_sentiment"] = None

absa_df.head()

,product_id,product_name,main_category,discount_percentage,discount_group,rating,sentence,aspect,aspect_sentiment
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,looks durable charging is fine toono complains...,battery,None
1,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,looks durable charging is fine toono complains...,quality,None
2,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,looks durable charging is fine toono complains...,performance,None
3,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,looks durable charging is fine toono complains...,durability,None
4,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories,64,50%+,4.2,the charging speed is slower than the original...,battery,None


In [21]:
nlp = spacy.load("en_core_web_sm")

In [27]:
# --- ABSA smoke test (SetFit pipeline) ---
# Wire up the pipeline package (notebook lives in data/, package lives in ../pipeline/)
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "data" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pipeline.absa import predict
from pipeline.preprocess import explode_sentences
from pipeline.aggregate import attach_predictions, aspect_sentiment_by_group, top_complaints

In [28]:
# Sanity check: does SetFit ABSA load and produce sensible predictions?
# First call downloads ~100MB into ../models/setfit_absa/ - subsequent calls are cached.
demo = [
    "The battery dies after two hours, very disappointing.",
    "Packaging was excellent and shipping was super fast.",
    "Great sound quality for the price.",
]
for sentence, spans in zip(demo, predict(demo)):
    print(sentence)
    for span in spans:
        print(f"  - {span['span']}: {span['polarity']}")

The battery dies after two hours, very disappointing.
  - battery: negative
Packaging was excellent and shipping was super fast.
  - Packaging: positive
  - shipping: positive
Great sound quality for the price.
  - quality: positive
  - price: positive


In [29]:
# End-to-end on a 10-review sample (first real ABSA run on the dataset)
sample = df.sample(10, random_state=42)
sentences_df = explode_sentences(sample)
print(f"{len(sentences_df)} sentences from {len(sample)} reviews")

preds = predict(sentences_df["sentence"].tolist())
spans_df = attach_predictions(sentences_df, preds)
print(f"{len(spans_df)} detected aspect spans after keyword mapping")
spans_df.head()

69 sentences from 10 reviews
38 detected aspect spans after keyword mapping


,product_id,product_name,main_category,discount_percentage,discount_group,rating,sentence,sentence_idx,span,polarity,aspect
0,B07P681N66,TP-Link AC600 600 Mbps WiFi Wireless Network U...,Computers&Accessories,45,30-50%,4.4,"i am using this on an old mac mini, since the ...",0,download speeds,positive,performance
1,B07P681N66,TP-Link AC600 600 Mbps WiFi Wireless Network U...,Computers&Accessories,45,30-50%,4.4,but even though i had never used dkms or iw be...,5,build tools,neutral,quality
2,B07P681N66,TP-Link AC600 600 Mbps WiFi Wireless Network U...,Computers&Accessories,45,30-50%,4.4,i then got the same download speed using this ...,6,download speed,neutral,performance
3,B07P681N66,TP-Link AC600 600 Mbps WiFi Wireless Network U...,Computers&Accessories,45,30-50%,4.4,i then got the same download speed using this ...,6,download speed,neutral,performance
4,B07P681N66,TP-Link AC600 600 Mbps WiFi Wireless Network U...,Computers&Accessories,45,30-50%,4.4,there is also a driver of realtek which you mi...,10,performance,positive,performance


In [30]:
# Dashboard-ready aggregations on the same 10-review sample
by_group = aspect_sentiment_by_group(spans_df)
complaints = top_complaints(spans_df, n=5)

print("Sentiment by aspect x discount_group:")
print(by_group)
print("\nTop complaints:")
print(complaints)

Sentiment by aspect x discount_group:
polarity       aspect discount_group  negative  neutral  positive  total  \
0             battery         15-30%         0        0         1      1   
1            delivery         30-50%         0        1         0      1   
2         performance         15-30%         1        0         1      2   
3         performance         30-50%         0        3         3      6   
4         performance           50%+         0        0         1      1   
5             quality         15-30%         0        0         1      1   
6             quality         30-50%         1        1         3      5   
7             quality           50%+         0        0         7      7   
8               sound           50%+         0        0         3      3   
9               value          0-15%         0        0         2      2   
10              value         15-30%         0        0         1      1   
11              value         30-50%         0    

In [31]:
# --- Build labeling CSV for polarity fine-tuning ---
# Runs the pipeline on a larger sample and writes predictions to a CSV
# you'll hand-correct. Edit the `polarity` column, then run:
#     python training/train_polarity.py
LABEL_SAMPLE_SIZE = 50

label_sample = df.sample(LABEL_SAMPLE_SIZE, random_state=7)
label_sentences = explode_sentences(label_sample)
label_preds = predict(label_sentences["sentence"].tolist())
label_spans = attach_predictions(label_sentences, label_preds)
print(f"{len(label_spans)} candidate rows for labeling")

labels_out = (
    label_spans[["sentence", "span", "polarity", "aspect"]]
    .rename(columns={"sentence": "text"})
    .assign(ordinal=0,
            polarity_predicted=lambda d: d["polarity"])
    [["text", "span", "polarity", "polarity_predicted", "aspect", "ordinal"]]
)

labels_path = PROJECT_ROOT / "training" / "labels.csv"
labels_path.parent.mkdir(parents=True, exist_ok=True)
labels_out.to_csv(labels_path, index=False)
print(f"Wrote {labels_path}")
print("Now open the CSV, correct the `polarity` column (positive/negative/neutral),"
      " delete rows you can't label, save it, then run:"
      "\n    python training/train_polarity.py")

261 candidate rows for labeling
Wrote c:\Users\appen\OneDrive\Área de Trabalho\nlp_project\price-pulse\training\labels.csv
Now open the CSV, correct the `polarity` column (positive/negative/neutral), delete rows you can't label, save it, then run:
    python training/train_polarity.py
